---
title: Searching with Namespaces and XPath
---

:::{error} Blank notebook
This section introduces XPath and provides more granular filter options with paths. For example, those who complete the section should be able to answer a question like "Now I need only titleInfo/title, not relatedItem/titleInfo/title". BS4 and simple ET were not able to express an answer to that question.
:::

In [2]:
# Load data
from pathlib import Path

DATA = Path('..', 'data')
OUT = Path('..', 'output')

EAD_sample = DATA / 'sample-ead-superior.xml'

# Start ElementTree
import xml.etree.ElementTree as ET

# Load data
tree = ET.parse(EAD_sample)
root = tree.getroot()

## Working with namespaces

Now, let's simplify the usage of namespaces. As you can see from the tags above, it can get tedious to use the full reference for each tag. When an XML with a namespace declaration is parsed by eTree, it prepends the associated namespace to each tag element. Thus, the `control` element in the EAD document here becomes `{http://ead3.archivists.org/schema/}control`.
This long name is called the qualified name or QName.
It can be a lot to type each time you reference an element, and it would be
efficient to shorten this to something shorter. For example, a consistent prefix.
That is possible using the ElementTree namespacing features, which allow you to create prefixes for any schema definitions or URIs.
In this case, for example, `ead:control` would be a useful shorthand. 

To do this, the etree module provides a namespace handler. To initialize namespaces, establish a dictionary, typically named `ns` (or something short and easy to remember), that will be passed into the parser:

In [3]:
ns = {
    'ead' : 'http://ead3.archivists.org/schema/'
}

Previously, a reference to the control element required the full QName.

In [5]:
control = root.find('{http://ead3.archivists.org/schema/}control')

print(control.tag,' -- ',control.attrib)

{http://ead3.archivists.org/schema/}control  --  {'countryencoding': 'iso3166-1', 'dateencoding': 'iso8601', 'langencoding': 'iso639-2b'}


ElementTree's namespacing feature allows the namespace dictionary to be
reference in `.find()`, `.findall()`, and some other functions.
Thus the prefix and element name can be used, greatly simplifying searches and element references. 

In [6]:
control = root.find('ead:control', ns)
print(control.tag,' -- ',control.attrib)

{http://ead3.archivists.org/schema/}control  --  {'countryencoding': 'iso3166-1', 'dateencoding': 'iso8601', 'langencoding': 'iso639-2b'}


Note that the parser still retains the full QName, but it is no longer needed in code references.

The utility of the prefix shorthand is clearer with nested elements.
Take a closer look with the `<titleproper>` element. In the [sample data](/part02/xml-01-basics-with-ET.ipynb#ead-ex01), `titleproper` appears on line 7.
A quick look in the tree shows that it is wrapped in the `titlestmt` element, which is in the `filedesc`, which is in `control`. These are all in the `ead` root element.
If you were to write a path that addressed `titleproper` with Qnames, it would look like

```
{http://ead3.archivists.org/schema/}ead
    {http://ead3.archivists.org/schema/}control 
        {http://ead3.archivists.org/schema/}filedesc
            {http://ead3.archivists.org/schema/}titlestmt
                {http://ead3.archivists.org/schema/}titleproper
```

Namespace prefixes significantly shorten this notation:

```
ead:ead / ead:control / ead:filedesc / ead:titlestmt / ead:titleproper
```

This much shorter construction, then, provides a valid `.find()` expression for the title information. Note that this construction uses the full ElementTree (referenced by `tree`) rather than the parsed root element. To get around that, it references
the implicit root element by using the dot (`.`) to indicate the "you are here" point,
which is the root.

In [15]:
titleproper = tree.find('./ead:control/ead:filedesc/ead:titlestmt/ead:titleproper', ns)
print(titleproper.tag,titleproper.text)

{http://ead3.archivists.org/schema/}titleproper A Finding Aid for the Superior Papers


:::{hint} Why are namespaces important?
QNames and namespaces may seem a bit fussy, but they are critical for
managing metadata since they allow for much more specific definition
of the spcial language structures that comprise metadata.
The highly specific name referencing reduces ambiguity, and it allows for high precision in defining what an element means.

For example, if you see `ead:control` in an XML tag,
it indicates all of the rules and usage requirements that are
defined in detail at the EAD specification: https://loc.gov/ead/EAD3taglib/.
:::

## Introducing XPath

XPath is a selector language that allows us to search for specific elements and attributes within the tree. The power of this language is that it allows us to select very precisely, and it also allows us to see multiple items at similar levels in the hierarchy or those that meet particular characteristics (such as having a particular attribute). 

Remember that while XML can be represented as a tree, any of the nodes, attributes, or embedded values in XML might also be represented by a path. [](#xml-tree-basic) illustrates a basic EAD hierarchy that maps onto the simplified EAD document in [](/part02/xml-01-basics-with-ET.ipynb#ead-ex01).

```{figure} /assets/xml-tree-basic.png
:label: xml-tree-basic
:alt: A graphic illustrating a sample EAD hierarchy with 4 levels and varied elements and attributes at each level, shown in a tree structure descending from a "root" EAD element

A graphic illustrating a sample EAD hierarchy with 4 levels and varied elements and attributes at each level, shown in a tree structure descending from an `ead` element at the "root" or base.
```

To see [](#xml-tree-basic) as a "tree," imagine turning the hierarchy shown upside down and then consider the `ead` element as a "root" from which the other elements branch out. The tree metaphor is commonly used in describing XML. This is useful in illustrating the hierarchical relationships, which reflects the inheritance relationships from element to element, and can illustrate "parent" (source nodes) and "child" (descending nodes) relationships. 

Another metaphor for locating information within the tree is a file path. In this representation of the structure, imagine notating each node from the top to the destination. Thus, we can create a specific address for each element in the structure. To address the root node, for example, use a slash and the name of the root element: `/ead`. To reference an entire level, for example everything in the `dsc` level, you might use a path expansion: `/ead/archdesc/dsc/*`. Individual attributes may be referenced by the `@` symbol: `/ead/archdesc[@level]`. This notation provides a specific way to address any element of the structure and reference it within a program. 

**XPath** syntax allows for the creation of patterns that can select particular addresses within the tree. This might be akin to an advanced path expansion or regular expression but for finding things in XML. Currently, ElementTree does not provide full support for XPath syntax, but it does allow for many queries that allow a script to select data from XML in powerful ways. 

Paths in XPath separate node elements with slashes (`/`). 

Elements are strung together with slashes in bewteen to indicate a path from the location of the query. Generally, any query beginning with a slash descends from the root node, but this is not always the case. When the root is not the source, elements begin from a _context node_. 

XPath selectors available in ElementTree include:

| Syntax | Meaning |
| ------ | ------ | 
| `tag` | Selects all child elements of the context node with the given `tag`. These can be used with namespace selectors as well, for example `{namespace}*` selects all tags in a given namespace, or `{*}tag` selects all matching tags in any (or no namespace). |
| `*` | Selects all child elements of a context node. |
| `.` | Selects the current node. |
| `//` | Selects all subelements, on all levels beneath the context element. Useful for matching elements or attributes across various branches of the hierarchy. | 
| `..` | Select a parent element of the context node. |
| `[@attrib]` | Selects all elements with the given attribute. In this case where the attribute name matches `attrib`. |
| `[@attrib='value']` | Selects elements for which the attribute is a given value. In this case where an element includes `attrib="value"`. Note that the value cannot contain quote marks. | 
| `[tag]` | Selects elements with a child that matches the `tag`. | 

Additional XPath expressions are possible. For reference, see the [Python ElementTree documentation](https://docs.python.org/3/library/xml.etree.elementtree.html#supported-xpath-syntax), and for more on XPath generally this [quick introduction to XPath from Library Carpentry](https://librarycarpentry.org/lc-webscraping/02-xpath/index.html).


### Exploring Xpath in ElementTree

Let's work with Xpath in a few examples. 

In [7]:
title = tree.find('ead:control/ead:filedesc/ead:titlestmt/ead:titleproper', ns)
print(title.tag, title.text)

{http://ead3.archivists.org/schema/}titleproper A Finding Aid for the Superior Papers


:::{warning} TODO: more examples with the sample data
Write more examples!!!

Then, in the subsequent `.findall()` examples and below,
use a real world example. The following uses the Day Papers example,
but consider changing out to the Jim Toy papers? 
:::

#### Using `.findall()`

Many elements will occur more than once in a given tree.
When looking multiple elements, the `.findall()` function, which operates similarly, is a better choice.
For example, the above list of all tags produced by `.iter()` shows multiple `part` elements. How would you select each of these?

In [ ]:
for title in tree.findall('ead:control/ead:filedesc/ead:titlestmt/ead:titleproper', ns):
    print(title.tag, title.text)

{http://ead3.archivists.org/schema/}titleproper Finding Aid for the William R. Day Collection day 
{http://ead3.archivists.org/schema/}titleproper William R. Day Collection


Using an XPath selector, we can write the above more efficiently. We can select all elements with a matching element tag using `.` followed by `//`. The `.` selects the current element (if we specify `tree` that is the root element in this case `ead`), and the double slash `//` selects any child element matching the element name supllied. Thus. `.//ead:titleproper` will select any `titleproper` elements in the file:

In [ ]:
for title in tree.findall('.//ead:titleproper', ns):
    print(title.tag, title.text)

{http://ead3.archivists.org/schema/}titleproper Finding Aid for the William R. Day Collection day 
{http://ead3.archivists.org/schema/}titleproper William R. Day Collection


Similarly, we could look for some of the elements in the collection contents using the `c` elements. For example, the `c01` tag (representing the "first level" of container objects), occurs multiple times and in this case represents the various series in the collection). Below, the loop uses `findall` to look for all `c01` elements in the current tree, prints the `id` attribute, tag name and list of attributes. Then, a `find` statement is extracts the text of the series description from the `scopecontent` note of the `c01` level and prints it. Finally, a second nested `findall()` looks for the `unittitle` of all `c02` elements within the series and prints a list of the folders or boxes in the series:  

In [ ]:
did_count = 0

for obj in tree.findall('.//ead:c01', ns):
    did_count += 1
    # extract the series id for the c01 element 
    print(f'Series id: {obj.attrib["id"]}\n', obj.tag, obj.attrib)

    # extract and print the paragraph in the scopecontent note for the series 
    scope = obj.find('.//ead:scopecontent/ead:p', ns)
    print(scope.text,'\n')

    # look through the siers and find the c02 second levels and their unittitles to see the various folders or subseries in the c01 level
    for item in obj.findall('.//ead:c02//ead:unittitle', ns):
        print(item.text)

Series id: aspace_ref1
 {http://ead3.archivists.org/schema/}c01 {'id': 'aspace_ref1', 'level': 'series'}
The Correspondence and Papers series contains correspondence and papers from William Day and various family members. 

William Day
1896
1897
1898 (3 folders)
1899
1900-1911
1920-1923
Undated
Biographical
Scrapbook
Family
Luther Day (Father)
Emily Spalding Day (Mother)
Louis Schaefer (Father-in-Law)
Other Members
Miscellaneous (3 folders)
Series id: aspace_ref18
 {http://ead3.archivists.org/schema/}c01 {'id': 'aspace_ref18', 'level': 'series'}
The Manuscripts series contains work by William Day and his son, Stephen Day. It also has a dissertation about William Day by Joseph McLean, and a folder of miscellaneous materials. 

William Day on McKinley
Stephan Day Notebook
McLean Dissertation on William Day (2 folders)
Miscellaneous
Series id: aspace_ref22
 {http://ead3.archivists.org/schema/}c01 {'id': 'aspace_ref22', 'level': 'series'}
The Newspaper series includes issues of the Univers

## Summary

This section drew on the basics of parsing XML with the ET module in Python to introduce and demonstrate the basic properties of XPath. XPath provides an efficient, path-like language to sort through, identify, and select elements within an XML structure with a powerful set of query filters.

Along with the section introduction XML, the sections on inspecting, modifying, constructing and writing, and validating, this provides a full toolbox that supports working with, analyzing, and modifying XML.
  